### How far does WRDS Datastream go?
Latest date available for each future in `configs/data.json`. Logging in asks for your WRDS username and password (and maybe Duo).

In [ ]:
import json

import wrds

universe = json.load(open("../configs/data.json"))["universe"]
codes = {v["contrcode"]: k for k, v in universe.items()}

db = wrds.Connection()

In [ ]:
df = db.raw_sql(f"""
    SELECT i.contrcode, max(f.date_) AS last_date
    FROM trdstrm.wrds_fut_contract f
    JOIN trdstrm.wrds_contract_info i USING (futcode)
    WHERE i.contrcode IN ({','.join(map(str, codes))})
    GROUP BY i.contrcode""")
db.close()

df["ticker"] = df.contrcode.map(codes)
df = df.sort_values("last_date")[["ticker", "last_date"]]
print("latest date overall:", df.last_date.max())
print("earliest last date: ", df.last_date.min())
df